In [ ]:
import torch
import torch.nn.functional as F
from transformers import PretrainedConfig
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer, DefaultDataCollator, DataCollatorForTokenClassification, AutoConfig

device = 'cpu'
GEMMA = 4 # num of tokens for speculative decoding
# STD_MODEL_PATH = '/Users/yingyao/Desktop/Code/GetHandsDirty.nosync/gz-data/Qwen2.5-0.5B-Instruct'
# TCH_MODEL_PATH = '/Users/yingyao/Desktop/Code/GetHandsDirty.nosync/gz-data/Qwen2.5-1.5B-Instruct'# GLM-4-9B-0414

std_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
tch_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
std_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct', use_fast=True, fix_mistral_regex=True)
tch_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', use_fast=True, fix_mistral_regex=True)


In [ ]:
print(len(std_tokenizer) == len(tch_tokenizer)) # prerequisitie: same tokenizer
tokenizer = std_tokenizer

True


In [5]:
inputs = tokenizer("The Qwen2 model family is", return_tensors="pt").to(device)
outputs = std_model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The Qwen2 model family is a multi-modal transformer-based language model, capable of handling various types of text data. Inspired by its capabilities, let's consider a simplified scenario where we are modeling the growth of a bacterial population in a controlled environment. The bacteria population grows exponentially over time.

Suppose that the initial population of bacteria is \( P_0 = 100 \) and it doubles every hour. After how many hours will the population reach \( P_{16} \)? To solve this problem, we can


In [30]:
def speculative_decoding(std_model, tch_model, input_ids, max_new_tokens=100, gemma=GEMMA, temperature=1.0):
    std_model.eval()
    tch_model.eval()
    current_ids = input_ids.clone()

    while current_ids.size(1) < max_new_tokens:
        draft_ids = current_ids.clone()
        draft_probs_list = []

        # Use student model to generate multiple tokens at once
        for _ in range(gemma):
            with torch.no_grad():
                std_logits = std_model(draft_ids).logits[:, -1, :]
                std_probs = F.softmax(std_logits / temperature, dim=-1) # vocab size probs
                draft_next_id = torch.multinomial(std_probs, num_samples=1)
                draft_probs_list.append(std_probs)
                draft_ids = torch.cat([draft_ids, draft_next_id], dim=-1)
                # print(draft_probs_list)

        draft_probs_tensor = torch.stack(draft_probs_list, dim=1)
        generated_draft_tokens = draft_ids[:, current_ids.size(1):]
        # print('draft_probs_tensor: ', draft_probs_tensor)
        # print('generated_draft_tokens: ', generated_draft_tokens)

        # Calculate teacher logits for those tokens
        with torch.no_grad():
            tch_logits = tch_model(draft_ids).logits[:, current_ids.size(1)-1 : -1, :]
            tch_probs_tensor = F.softmax(tch_logits / temperature, dim=-1) # vocab size probs
        # print('tch_probs_tensor: ', tch_probs_tensor)

        # Use rejected sampling to accept draft / reject it and resample
        n_accepted = 0
        correction_token = None
        for i in range(gemma):
            token_id = generated_draft_tokens[0, i]
            q_prob = draft_probs_tensor[0, i, token_id]
            p_prob = tch_probs_tensor[0, i, token_id]

            acceptance_ratio = min(1.0, (p_prob / q_prob).item()) # for the tokens selected from the draft model, how likely we decides to accept it?

            if torch.rand(1).item() < acceptance_ratio:
                n_accepted += 1
            else:
                # re-select the tokens from entire vocabs based on corrected dist, just like picking markables from a bag with replacement
                p_dist = tch_probs_tensor[0, i, :]
                q_dist = draft_probs_tensor[0, i, :]

                # renormalize the dist based on corrected dist
                corrected_dist = torch.clamp(p_dist - q_dist, min=0.0)
                corrected_dist = corrected_dist / corrected_dist.sum() # Normalize
                correction_token = torch.multinomial(corrected_dist.unsqueeze(0), num_samples=1)
                break

        # Update sequence
        accepted_tokens = generated_draft_tokens[:, :n_accepted]
        print('accepted_tokens, ', accepted_tokens)
        if correction_token is not None:
            current_ids = torch.cat([current_ids, accepted_tokens, correction_token], dim=-1)
        else:
            # we accept all draft tokens! and we get extra bonus token from teacher model! 
            with torch.no_grad():
                final_logits = tch_model(draft_ids).logits[:, -1, :]
                tch_probs = F.softmax(final_logits / temperature, dim=-1) # vocab size probs
                final_token = torch.multinomial(tch_probs, num_samples=1)
                print('extra final token from teacher model, ', final_token)
            current_ids = torch.cat([current_ids, accepted_tokens, final_token], dim=-1)
            
    return current_ids


In [14]:
torch.rand(1).item()

0.2645567059516907

In [31]:
current_ids = speculative_decoding(std_model, tch_model, inputs['input_ids'], max_new_tokens=100)

accepted_tokens,  tensor([], size=(1, 0), dtype=torch.int64)
accepted_tokens,  tensor([[448]])
accepted_tokens,  tensor([[16, 15, 15]])
accepted_tokens,  tensor([], size=(1, 0), dtype=torch.int64)
accepted_tokens,  tensor([[52977]])
accepted_tokens,  tensor([[220,  16,  13,  20]])
extra final token from teacher model,  tensor([[33]])
accepted_tokens,  tensor([], size=(1, 0), dtype=torch.int64)
accepted_tokens,  tensor([[1467,   11,  323]])
accepted_tokens,  tensor([[ 1331, 39586,  7699,    13]])
extra final token from teacher model,  tensor([[3197]])
accepted_tokens,  tensor([[ 498, 1787,  279, 1207]])
extra final token from teacher model,  tensor([[16948]])
accepted_tokens,  tensor([[12]])
accepted_tokens,  tensor([], size=(1, 0), dtype=torch.int64)
accepted_tokens,  tensor([], size=(1, 0), dtype=torch.int64)
accepted_tokens,  tensor([[ 11, 279]])
accepted_tokens,  tensor([[40155,   264]])
accepted_tokens,  tensor([], size=(1, 0), dtype=torch.int64)
accepted_tokens,  tensor([], size=(

In [32]:
print(tokenizer.decode(current_ids[0], skip_special_tokens=True))

The Qwen2 model family is trained with 100M synthetic visuals, 1.5B synthetic text, and real-speech audio. When you open the Qwen-BaseAI webpage, the platform captures a webcam to record your speech. AI listens to the voice and proposes matching text that can be humanized and added to other media.
Qwen-BaseAI could generate out-of-context interpretations and simplify styles from other texts to make them more accessible. Please see the examples:

